# 02 · CNN entraîné de zéro

Nous comparons une référence simple, l'augmentation de données puis le dropout. L'idée est de modifier un seul facteur à la fois.

In [ ]:
from pathlib import Path
import sys
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/SceneSense')]
ROOT = next((p for p in candidates if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the SceneSense project folder in Colab or Jupyter')
sys.path.insert(0, str(ROOT / 'src'))
from scenesense.config import Settings, CLASSES
settings = Settings(data_dir=ROOT / "data/intel-image-classification", output_dir=ROOT / "results")
print("Project:", ROOT)
print("Dataset:", settings.data_dir)

## Architecture

Trois convolutions 3×3 (32, 64, 128 filtres), trois max-pooling 2×2, GlobalAveragePooling, Dense(128), Softmax(6).

In [ ]:
from scenesense.models import build_custom_cnn, compile_model
model = compile_model(build_custom_cnn(image_size=settings.size))
model.summary()

## Entraîner E1, E2 et E3

Le meilleur checkpoint est enregistré selon la perte de validation, indépendamment des données de test.

In [ ]:
from scenesense.data import read_manifest
from scenesense.training import train_experiment
assert settings.manifest_path.exists(), "Run notebook 01 first to prepare the fixed split"
cnn_results = []
for experiment in ("cnn_baseline", "cnn_augmentation", "cnn_dropout"):
    cnn_results.append(train_experiment(experiment, settings))

In [ ]:
import pandas as pd
display(pd.DataFrame(cnn_results)[["experiment", "validation_accuracy", "validation_loss", "epochs_completed", "duration_seconds"]])

## Commentaire

Comparer E1 à E2 pour isoler l'augmentation de données, puis E2 à E3 pour isoler le dropout. Si la validation ne s'améliore pas, expliquer ce résultat au lieu de le masquer.